In [ ]:
import yfinance as yf

def analyze_stock(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info

    if not info or "regularMarketPrice" not in info:
       print(f"\n'{ticker}' doesn't appear to be a valid stock ticker or no data is available. Please try a company ticker like AAPL or MSFT.")
       return

    reasons = []

    pe_score = 0
    pe = info.get("trailingPE", None)
    if pe is not None:
        if pe < 15:
            pe_score += 1
            reasons.append(f"✓ Very cheap valuation (P/E: {pe:.1f})")
        elif pe < 25:
            pe_score += 0.75
            reasons.append(f"✓ Reasonable valuation (P/E: {pe:.1f})")
        elif pe < 40:
            pe_score += 0.5
            reasons.append(f"~ Premium valuation (P/E: {pe:.1f}) — priced for growth")
        else:
            pe_score += 0
            reasons.append(f"✗ Very expensive valuation (P/E: {pe:.1f})")
    else:
        reasons.append("? P/E ratio not available")

    debt_score = 0
    debt = info.get("debtToEquity", None)
    if debt is not None:
        debt = debt/100
        if debt < 0.5:
            debt_score += 1
            reasons.append(f"✓ Very low debt ({debt:.2f}) — financially safe")
        elif debt < 1.0:
            debt_score += 0.75
            reasons.append(f"✓ Healthy debt ({debt:.2f})")
        elif debt < 2.0:
            debt_score += 0.5
            reasons.append(f"~ Moderate debt ({debt:.2f})")
        else:
            debt_score += 0
            reasons.append(f"✗ High debt ({debt:.2f}) — risky")
    else:
        reasons.append("? Debt data not available")

    growth_score = 0
    rev_growth = info.get("revenueGrowth", None)
    if rev_growth is not None:
        if rev_growth > 0.20:
            growth_score += 1
            reasons.append(f"✓ Exceptional revenue growth ({rev_growth:.0%})")
        elif rev_growth > 0.10:
            growth_score += 0.75
            reasons.append(f"~ Strong revenue growth ({rev_growth:.0%})")
        elif rev_growth > 0:
            growth_score += 0.5
            reasons.append(f"~ Slow but positive revenue growth ({rev_growth:.0%})")
        else:
            growth_score += 0
            reasons.append("✗ Revenue shrinking")
    else:
        reasons.append("? Revenue growth data unavailable")

    analyst_score = 0
    rec = info.get("recommendationKey", None)
    if rec is not None:
        if rec in ["strong_buy"]:
            analyst_score += 1
            reasons.append(f"✓ Analysts recommend: {rec.upper()}")
        elif rec == "buy":
            analyst_score += 0.75
            reasons.append(f"✓ Analysts recommend: {rec.upper()}")
        elif rec == "hold":
            analyst_score += 0.5
            reasons.append("~ Analysts say HOLD")
        else:
            analyst_score += 0
            reasons.append(f"✗ Analysts say: {rec}")
    else:
        reasons.append("? Analyst recommendation not available")

    margins_score = 0
    margins = info.get("profitMargins", None)
    if margins is not None:
        if margins > 0.20:
           margins_score += 1
           reasons.append(f"✓ Strong profit margins ({margins:.0%})")
        elif margins and margins > 0.10:
           margins_score += 0.75
           reasons.append(f"~ Decent profit margins ({margins:.0%})")
        elif margins and margins > 0:
           margins_score += 0.5
           reasons.append(f"✗ Thin profit margins ({margins:.0%})")
        else:
           margins_score = 0
           reasons.append(f"✗ Negative profit margins ({margins:.0%}) — company is losing money")
    else:
        # Corrected: 'score' was being used here instead of initializing margins_score
        margins_score = 0 # Initialize if data not available, or keep it 0 as it was
        reasons.append("? Profit margin data not available")

    # Weighted final score
    weighted_score = (
        pe_score * 0.25 +
        debt_score * 0.15 +
        growth_score * 0.20 +
        margins_score * 0.25 +
        analyst_score * 0.15
    )

    # Verdict
    if weighted_score >= 0.65:
        verdict = "BUY"
    elif weighted_score >= 0.40:
        verdict = "HOLD"
    else:
        verdict = "AVOID"

    # Print output
    print(f"\n{'='*45}")
    print(f"  EQUITY ANALYSIS: {ticker.upper()}")
    print(f"  {info.get('longName', ticker)}")
    print(f"{'='*45}")
    for r in reasons:
        print(f"  {r}")
    print(f"{'='*45}")
    print(f"  WEIGHTED SCORE: {weighted_score:.2f}/1.00")
    print(f"  VERDICT: {verdict}")
    print(f"{'='*45}\n")
ticker = input("Enter a stock ticker: ")
analyze_stock(ticker)